In [7]:
import polars as pl
import polars_st as st
from polars_st import geom, geometry

In [5]:
data = pl.scan_parquet("../data/processed/firms_noaa20.parquet")

In [6]:
data.head().collect()

latitude,longitude,bright_ti4,scan,track,acq_date,acq_time,satellite,instrument,confidence,version,bright_ti5,frp,daynight,acquired_at_utc
f64,f64,f64,f64,f64,str,i64,str,str,str,str,f64,f64,str,"datetime[μs, UTC]"
-8.79831,140.69357,331.79,0.44,0.46,"""2026-08-24""",349,"""N20""","""VIIRS""","""n""","""2.0NRT""",295.91,3.73,"""D""",2026-08-24 03:49:00 UTC
-8.71738,140.74724,330.69,0.43,0.46,"""2026-08-24""",349,"""N20""","""VIIRS""","""n""","""2.0NRT""",290.74,1.54,"""D""",2026-08-24 03:49:00 UTC
-8.34927,138.0038,336.22,0.33,0.55,"""2026-08-24""",349,"""N20""","""VIIRS""","""n""","""2.0NRT""",302.29,5.4,"""D""",2026-08-24 03:49:00 UTC
-8.3187,137.76657,332.43,0.34,0.56,"""2026-08-24""",349,"""N20""","""VIIRS""","""n""","""2.0NRT""",299.93,2.74,"""D""",2026-08-24 03:49:00 UTC
-8.30191,137.89424,338.57,0.33,0.55,"""2026-08-24""",349,"""N20""","""VIIRS""","""n""","""2.0NRT""",301.05,4.29,"""D""",2026-08-24 03:49:00 UTC


In [16]:
data = data.with_columns(
    geometry=st.point(
        pl.concat_arr("longitude", "latitude"),
        srid=4326
    )
)

prov = st.read_file("../data/boundaries/geoBoundaries-IDN-ADM1-provinces.geojson")
prov = prov.select(
    pl.col("shapeName").alias("province"),
    "geometry"
)

/tmp/ipykernel_233868/72675481.py:8: FutureWarning: from_arrow(<ArrowStreamExportable>) will return a Series instead of a DataFrame in 2.0. To avoid this warning, pass the ArrowStreamExportable to either `pl.DataFrame` or `pl.Series` instead based on your desired output type.
  prov = st.read_file("../data/boundaries/geoBoundaries-IDN-ADM1-provinces.geojson")


In [17]:
print(prov.schema)
print(prov.head())

Schema({'province': String, 'geometry': Binary})
shape: (5, 2)
┌────────────────────┬─────────────────────────────────┐
│ province           ┆ geometry                        │
│ ---                ┆ ---                             │
│ str                ┆ binary                          │
╞════════════════════╪═════════════════════════════════╡
│ Bali               ┆ b"\x01\x06\x00\x00\x20\xe6\x10… │
│ West Nusa Tenggara ┆ b"\x01\x06\x00\x00\x20\xe6\x10… │
│ Banten             ┆ b"\x01\x06\x00\x00\x20\xe6\x10… │
│ Central Java       ┆ b"\x01\x06\x00\x00\x20\xe6\x10… │
│ West Java          ┆ b"\x01\x03\x00\x00\x20\xe6\x10… │
└────────────────────┴─────────────────────────────────┘


In [27]:
joined = (data.collect()
          .st.sjoin(
                prov,
                how="left",
                predicate="contains"
            )
          .drop("geometry_right")
          )

/tmp/ipykernel_233868/2806564407.py:2: DeprecationWarning: the default behavior of `how='horizontal'` for `concat` is deprecated and will require equal heights in the next breaking release. Use `how='horizontal_extend'` to keep the current behavior.
(Deprecated in version 1.42.1)
  .st.sjoin(


In [28]:
print(joined.schema)
print(joined.head())

Schema({'latitude': Float64, 'longitude': Float64, 'bright_ti4': Float64, 'scan': Float64, 'track': Float64, 'acq_date': String, 'acq_time': Int64, 'satellite': String, 'instrument': String, 'confidence': String, 'version': String, 'bright_ti5': Float64, 'frp': Float64, 'daynight': String, 'acquired_at_utc': Datetime(time_unit='us', time_zone='UTC'), 'geometry': Binary, 'province': String})
shape: (5, 17)
┌──────────┬───────────┬────────────┬──────┬───┬──────────┬──────────────┬──────────────┬──────────┐
│ latitude ┆ longitude ┆ bright_ti4 ┆ scan ┆ … ┆ daynight ┆ acquired_at_ ┆ geometry     ┆ province │
│ ---      ┆ ---       ┆ ---        ┆ ---  ┆   ┆ ---      ┆ utc          ┆ ---          ┆ ---      │
│ f64      ┆ f64       ┆ f64        ┆ f64  ┆   ┆ str      ┆ ---          ┆ binary       ┆ str      │
│          ┆           ┆            ┆      ┆   ┆          ┆ datetime[μs, ┆              ┆          │
│          ┆           ┆            ┆      ┆   ┆          ┆ UTC]         ┆            

In [31]:
indo_only = pl.read_parquet("../data/processed/firms_noaa20_indonesia.parquet")

In [33]:
indo_only.head()

latitude,longitude,bright_ti4,scan,track,acq_date,acq_time,satellite,instrument,confidence,version,bright_ti5,frp,daynight,acquired_at_utc,geometry,province,kabupaten_kota
f64,f64,f64,f64,f64,str,i64,str,str,str,str,f64,f64,str,"datetime[μs, UTC]",binary,str,str
-8.79831,140.69357,331.79,0.44,0.46,"""2026-08-24""",349,"""N20""","""VIIRS""","""n""","""2.0NRT""",295.91,3.73,"""D""",2026-08-24 03:49:00 UTC,"b""\x01\x01\x00\x00\x20\xe6\x10\x00\x00\x93o\xb6\xb91\x96a@\xb8#\x9c\x16\xbc\x98!\xc0""","""Papua""","""Merauke"""
-8.71738,140.74724,330.69,0.43,0.46,"""2026-08-24""",349,"""N20""","""VIIRS""","""n""","""2.0NRT""",290.74,1.54,"""D""",2026-08-24 03:49:00 UTC,"b""\x01\x01\x00\x00\x20\xe6\x10\x00\x00kH\xdcc\xe9\x97a@\xe5\x9bmnLo!\xc0""","""Papua""","""Merauke"""
-8.34927,138.0038,336.22,0.33,0.55,"""2026-08-24""",349,"""N20""","""VIIRS""","""n""","""2.0NRT""",302.29,5.4,"""D""",2026-08-24 03:49:00 UTC,"b""\x01\x01\x00\x00\x20\xe6\x10\x00\x002w-!\x1f@a@\xa6\xf2v\x84\xd3\xb2\x20\xc0""","""Papua""","""Merauke"""
-8.3187,137.76657,332.43,0.34,0.56,"""2026-08-24""",349,"""N20""","""VIIRS""","""n""","""2.0NRT""",299.93,2.74,"""D""",2026-08-24 03:49:00 UTC,"b""\x01\x01\x00\x00\x20\xe6\x10\x00\x00\x08\x03\xcf\xbd\x878a@lxz\xa5,\xa3\x20\xc0""","""Papua""","""Merauke"""
-8.30191,137.89424,338.57,0.33,0.55,"""2026-08-24""",349,"""N20""","""VIIRS""","""n""","""2.0NRT""",301.05,4.29,"""D""",2026-08-24 03:49:00 UTC,"b""\x01\x01\x00\x00\x20\xe6\x10\x00\x00\xcdX4\x9d\x9d<a@\xb4\xab\x90\xf2\x93\x9a\x20\xc0""","""Papua""","""Merauke"""


In [36]:
daily_kabkot = pl.read_parquet("../data/analytics/daily_kabupaten_kota.parquet")
daily_prov = pl.read_parquet("../data/analytics/daily_province.parquet")
firms30d = pl.read_parquet("../data/analytics/firms_30d.parquet")

In [43]:
print(daily_kabkot.shape)
display(daily_kabkot.head())
print(daily_prov.shape)
display(daily_prov.head())
print(firms30d.shape)
display(firms30d.head())

(6947, 7)


acq_date,province,kabupaten_kota,hotspot_count,high_confidence_count,total_frp,max_frp
str,str,str,u32,u32,f64,f64
"""2026-07-26""","""Papua""","""Merauke""",243,1,2161.03,132.06
"""2026-07-26""","""West Kalimantan""","""Sanggau""",138,0,2257.02,198.95
"""2026-07-26""","""West Kalimantan""","""Ketapang""",63,0,636.93,181.99
"""2026-07-26""","""West Kalimantan""","""Kayong Utara""",62,1,317.4,18.95
"""2026-07-26""","""Papua""","""Mappi""",59,0,465.21,30.18


(939, 6)


acq_date,province,hotspot_count,high_confidence_count,total_frp,max_frp
str,str,u32,u32,f64,f64
"""2026-07-26""","""West Kalimantan""",441,2,5424.03,198.95
"""2026-07-26""","""Papua""",354,1,2989.11,132.06
"""2026-07-26""","""Central Kalimantan""",144,7,1096.95,76.31
"""2026-07-26""","""East Java""",91,1,434.56,34.02
"""2026-07-26""","""Central Java""",68,0,237.98,29.55


(98245, 18)


latitude,longitude,bright_ti4,scan,track,acq_date,acq_time,satellite,instrument,confidence,version,bright_ti5,frp,daynight,acquired_at_utc,geometry,province,kabupaten_kota
f64,f64,f64,f64,f64,str,i64,str,str,str,str,f64,f64,str,"datetime[μs, UTC]",binary,str,str
-8.81859,140.97943,328.55,0.42,0.45,"""2026-07-26""",434,"""N20""","""VIIRS""","""n""","""2.0NRT""",294.56,2.43,"""D""",2026-07-26 04:34:00 UTC,"b""\x01\x01\x00\x00\x20\xe6\x10\x00\x00\x15W\x95}W\x9fa@P\xaa}:\x1e\xa3!\xc0""","""Papua""","""Merauke"""
-8.81766,140.95972,341.96,0.42,0.45,"""2026-07-26""",434,"""N20""","""VIIRS""","""n""","""2.0NRT""",296.72,4.42,"""D""",2026-07-26 04:34:00 UTC,"b""\x01\x01\x00\x00\x20\xe6\x10\x00\x00&\xaa\xb7\x06\xb6\x9ea@\xa6~\xdeT\xa4\xa2!\xc0""","""Papua""","""Merauke"""
-8.81443,140.97874,328.47,0.42,0.45,"""2026-07-26""",434,"""N20""","""VIIRS""","""n""","""2.0NRT""",294.4,2.72,"""D""",2026-07-26 04:34:00 UTC,"b""\x01\x01\x00\x00\x20\xe6\x10\x00\x00/i\x8c\xd6Q\x9fa@7\xc3\x0d\xf8\xfc\xa0!\xc0""","""Papua""","""Merauke"""
-8.80832,140.98946,329.22,0.42,0.45,"""2026-07-26""",434,"""N20""","""VIIRS""","""n""","""2.0NRT""",294.32,5.82,"""D""",2026-07-26 04:34:00 UTC,"b""\x01\x01\x00\x00\x20\xe6\x10\x00\x00h\x96\x04\xa8\xa9\x9fa@\xcbgy\x1e\xdc\x9d!\xc0""","""Papua""","""Merauke"""
-8.70118,140.58792,339.31,0.4,0.44,"""2026-07-26""",434,"""N20""","""VIIRS""","""n""","""2.0NRT""",293.06,3.86,"""D""",2026-07-26 04:34:00 UTC,"b""\x01\x01\x00\x00\x20\xe6\x10\x00\x00B\x95\x9a=\xd0\x92a@\xf47\xa1\x10\x01g!\xc0""","""Papua""","""Merauke"""


In [47]:
firms30d.filter(
    pl.col("acq_date") == "2026-08-23",
    pl.col("province") == "East Kalimantan"
).height

306

In [48]:
(
    daily_kabkot
    .filter(
        pl.col("acq_date") == "2026-08-23"
    )
    .sort(
        "hotspot_count",
        descending=True,
    )
    .head(10)
)

acq_date,province,kabupaten_kota,hotspot_count,high_confidence_count,total_frp,max_frp
str,str,str,u32,u32,f64,f64
"""2026-08-23""","""Papua""","""Merauke""",560,3,3713.1,75.77
"""2026-08-23""","""South Sumatra""","""Ogan Komering Ilir""",279,43,5412.3,126.32
"""2026-08-23""","""West Kalimantan""","""Ketapang""",227,4,1442.2,198.53
"""2026-08-23""","""Papua""","""Mappi""",180,0,1938.99,149.06
"""2026-08-23""","""East Kalimantan""","""Berau""",132,0,1716.87,163.78
"""2026-08-23""","""South Sumatra""","""Musi Banyuasin""",124,8,1064.05,67.14
"""2026-08-23""","""Central Kalimantan""","""Kotawaringin Timur""",111,1,286.41,15.0
"""2026-08-23""","""Maluku""","""Seram Bagian Timur""",105,0,181.56,5.71
"""2026-08-23""","""Riau""","""Indragiri Hilir""",104,9,700.97,54.58
